## Evaluate HanLP + Presidio on data_person_1000_zh\n
\n
This notebook runs your custom HanLP-backed Presidio analyzer against the Chinese test dataset and writes a summary JSON report.

In [1]:
from pathlib import Path
from collections import Counter
import json
import time

from presidio_analyzer import AnalyzerEngine, RecognizerRegistry

# Make sure we can import files from the project folder
ROOT = Path.cwd()
PROJECT_DIR = ROOT / "pii_detection_test"
if not PROJECT_DIR.exists():
    # If notebook is opened from inside pii_detection_test itself
    PROJECT_DIR = ROOT

import sys
if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))

from hanlp_engine import HanLPNlpEngine, HanLPRecognizer, SUPPORTED_ENTITIES

/Users/nikoli/Desktop/IC_LLM_API_GW/pii_detection_test/venv/lib/python3.11/site-packages/torch/cuda/__init__.py:51: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]
/Users/nikoli/Desktop/IC_LLM_API_GW/pii_detection_test/venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
INPUT_PATH = PROJECT_DIR / "pii_test_data" / "data_person_1000_zh.json"
OUTPUT_PATH = PROJECT_DIR / "pii_test_data" / "data_person_1000_zh_test_result.json"

print("Input:", INPUT_PATH)
print("Output:", OUTPUT_PATH)

with open(INPUT_PATH, "r", encoding="utf-8") as f:
    data = json.load(f)

print("Loaded records:", len(data))

Input: /Users/nikoli/Desktop/IC_LLM_API_GW/pii_detection_test/pii_test_data/data_person_1000_zh.json
Output: /Users/nikoli/Desktop/IC_LLM_API_GW/pii_detection_test/pii_test_data/data_person_1000_zh_test_result.json
Loaded records: 975


In [3]:
registry = RecognizerRegistry(supported_languages=["zh"])
registry.add_recognizer(HanLPRecognizer(supported_language="zh"))

analyzer = AnalyzerEngine(
    nlp_engine=HanLPNlpEngine(),
    registry=registry,
    supported_languages=["zh"],
)

print("Analyzer ready.")

Analyzer ready.


/Users/nikoli/Desktop/IC_LLM_API_GW/pii_detection_test/venv/lib/python3.11/site-packages/torch/_utils.py:831: UserWarning: TypedStorage is deprecated. It will be removed in the future and UntypedStorage will be the only storage class. This should only matter to you if you are using storages directly.  To access UntypedStorage directly, use tensor.untyped_storage() instead of tensor.storage()
  return self.fget.__get__(instance, owner)()


In [6]:
start = time.time()

entity_counter = Counter()
records_with_hits = 0
records_without_hits = 0
total_entities_detected = 0
sample_outputs = []

for i, row in enumerate(data, 1):
    text = row.get("naturalParagraph", "")
    if not text:
        continue

    results = analyzer.analyze(text=text, language="zh", entities=SUPPORTED_ENTITIES)

    if results:
        records_with_hits += 1
        total_entities_detected += len(results)
        for r in results:
            entity_counter[r.entity_type] += 1

        if len(sample_outputs) < 5:
            sample_outputs.append({
                "index": i,
                "name": row.get("name"),
                "count": len(results),
                "entities": [
                    {
                        "type": r.entity_type,
                        "text": text[r.start:r.end],
                        "start": r.start,
                        "end": r.end,
                        "score": round(float(r.score), 3),
                    }
                    for r in sorted(results, key=lambda x: x.start)
                ],
            })
    else:
        records_without_hits += 1

elapsed_seconds = round(time.time() - start, 2)

summary = {
    "dataset": str(INPUT_PATH),
    "records_total": len(data),
    "records_with_hits": records_with_hits,
    "records_without_hits": records_without_hits,
    "total_entities_detected": total_entities_detected,
    "entity_type_counts": dict(entity_counter),
    "sample_outputs": sample_outputs,
    "elapsed_seconds": elapsed_seconds,
}

with open(OUTPUT_PATH, "w", encoding="utf-8") as f:
    json.dump(summary, f, ensure_ascii=False, indent=2)

summary

/Users/nikoli/Desktop/IC_LLM_API_GW/pii_detection_test/venv/lib/python3.11/site-packages/torch/nn/modules/module.py:1527: FutureWarning: `encoder_attention_mask` is deprecated and will be removed in version 4.55.0 for `ElectraSelfAttention.forward`.
  return forward_call(*args, **kwargs)


{'dataset': '/Users/nikoli/Desktop/IC_LLM_API_GW/pii_detection_test/pii_test_data/data_person_1000_zh.json',
 'records_total': 975,
 'records_with_hits': 975,
 'records_without_hits': 0,
 'total_entities_detected': 20664,
 'entity_type_counts': {'PERSON': 6330,
  'PHONE_NUMBER': 12762,
  'FINANCIAL': 1561,
  'DATE_TIME': 11},
 'sample_outputs': [{'index': 1,
   'name': '白雅宁',
   'count': 20,
   'entities': [{'type': 'PERSON',
     'text': '白',
     'start': 0,
     'end': 1,
     'score': 0.85},
    {'type': 'PERSON', 'text': '雅', 'start': 1, 'end': 2, 'score': 0.85},
    {'type': 'PERSON', 'text': '宁', 'start': 2, 'end': 3, 'score': 0.85},
    {'type': 'PHONE_NUMBER',
     'text': '0',
     'start': 90,
     'end': 91,
     'score': 0.85},
    {'type': 'PHONE_NUMBER',
     'text': '0',
     'start': 92,
     'end': 93,
     'score': 0.85},
    {'type': 'PHONE_NUMBER',
     'text': '3',
     'start': 93,
     'end': 94,
     'score': 0.85},
    {'type': 'PHONE_NUMBER',
     'text': '1'

In [7]:
print("Saved summary to:", OUTPUT_PATH)
print("records_total:", summary["records_total"])
print("records_with_hits:", summary["records_with_hits"])
print("records_without_hits:", summary["records_without_hits"])
print("total_entities_detected:", summary["total_entities_detected"])
print("elapsed_seconds:", summary["elapsed_seconds"])
print("entity_type_counts:", summary["entity_type_counts"])

Saved summary to: /Users/nikoli/Desktop/IC_LLM_API_GW/pii_detection_test/pii_test_data/data_person_1000_zh_test_result.json
records_total: 975
records_with_hits: 975
records_without_hits: 0
total_entities_detected: 20664
elapsed_seconds: 26.51
entity_type_counts: {'PERSON': 6330, 'PHONE_NUMBER': 12762, 'FINANCIAL': 1561, 'DATE_TIME': 11}


In [8]:
# Show first sample output in a readable format
if summary["sample_outputs"]:
    first = summary["sample_outputs"][0]
    print("Sample index:", first["index"])
    print("Name:", first["name"])
    print("Detected entity count:", first["count"])
    for e in first["entities"][:20]:
        print(e)

Sample index: 1
Name: 白雅宁
Detected entity count: 20
{'type': 'PERSON', 'text': '白', 'start': 0, 'end': 1, 'score': 0.85}
{'type': 'PERSON', 'text': '雅', 'start': 1, 'end': 2, 'score': 0.85}
{'type': 'PERSON', 'text': '宁', 'start': 2, 'end': 3, 'score': 0.85}
{'type': 'PHONE_NUMBER', 'text': '0', 'start': 90, 'end': 91, 'score': 0.85}
{'type': 'PHONE_NUMBER', 'text': '0', 'start': 92, 'end': 93, 'score': 0.85}
{'type': 'PHONE_NUMBER', 'text': '3', 'start': 93, 'end': 94, 'score': 0.85}
{'type': 'PHONE_NUMBER', 'text': '1', 'start': 94, 'end': 95, 'score': 0.85}
{'type': 'PHONE_NUMBER', 'text': '8', 'start': 96, 'end': 97, 'score': 0.85}
{'type': 'PHONE_NUMBER', 'text': '0', 'start': 97, 'end': 98, 'score': 0.85}
{'type': 'PHONE_NUMBER', 'text': '0', 'start': 98, 'end': 99, 'score': 0.85}
{'type': 'PHONE_NUMBER', 'text': '8', 'start': 99, 'end': 100, 'score': 0.85}
{'type': 'PHONE_NUMBER', 'text': '2', 'start': 100, 'end': 101, 'score': 0.85}
{'type': 'PHONE_NUMBER', 'text': '7', 'start'

## 8. Evaluate results — Precision / Recall / F-score (same style as notebook 4)

Derives ground-truth spans from the structured JSON fields (`name`, `emailAddress`,
`phoneNumbers`, `location`, `idCardNumbers`, `doctor`), compares them against
the HanLP analyzer predictions, then plots per-entity Precision / Recall / F₂-score.

In [4]:
from collections import defaultdict
from typing import List, Tuple, Dict

# ── Field → Presidio entity type ───────────────────────────────────────────
FIELD_ENTITY_MAP = {
    "name":          "PERSON",
    "doctor":        "PERSON",
    "emailAddress":  "EMAIL_ADDRESS",
    "phoneNumbers":  "PHONE_NUMBER",
    "location":      "LOCATION",
    "idCardNumbers": "ID",
}

def find_all_spans(text: str, value: str) -> List[Tuple[int, int]]:
    spans, start = [], 0
    while True:
        idx = text.find(value, start)
        if idx == -1:
            break
        spans.append((idx, idx + len(value)))
        start = idx + 1
    return spans

def overlap(a_start, a_end, b_start, b_end) -> bool:
    return a_start < b_end and b_start < a_end

# ── Compute TP / FP / FN per entity type ───────────────────────────────────
tp = defaultdict(int)
fp = defaultdict(int)
fn = defaultdict(int)

for row in data:
    text = row.get("naturalParagraph", "")
    if not text:
        continue

    # Ground truth spans: [(start, end, entity_type), ...]
    gt_spans = []
    for field, etype in FIELD_ENTITY_MAP.items():
        raw = row.get(field)
        if raw is None:
            continue
        value = str(raw).strip()
        if value:
            for s, e in find_all_spans(text, value):
                gt_spans.append((s, e, etype))

    # Predicted spans
    pred_results = analyzer.analyze(text=text, language="zh", entities=SUPPORTED_ENTITIES)
    pred_spans = [(r.start, r.end, r.entity_type) for r in pred_results]

    # Match predictions to ground truth (per entity type, with char overlap)
    matched_gt = set()
    for ps, pe, ptype in pred_spans:
        matched = False
        for gi, (gs, ge, gtype) in enumerate(gt_spans):
            if gi in matched_gt:
                continue
            if ptype == gtype and overlap(ps, pe, gs, ge):
                tp[ptype] += 1
                matched_gt.add(gi)
                matched = True
                break
        if not matched:
            fp[ptype] += 1

    for gi, (gs, ge, gtype) in enumerate(gt_spans):
        if gi not in matched_gt:
            fn[gtype] += 1

# ── Scores ──────────────────────────────────────────────────────────────────
BETA = 2
entity_types = sorted(set(list(tp.keys()) + list(fn.keys())))

scores = {}
for e in entity_types:
    _tp, _fp, _fn = tp[e], fp[e], fn[e]
    precision = _tp / (_tp + _fp) if (_tp + _fp) > 0 else 0.0
    recall    = _tp / (_tp + _fn) if (_tp + _fn) > 0 else 0.0
    denom     = (1 + BETA**2) * precision + BETA**2 * recall
    f_score   = (1 + BETA**2) * precision * recall / denom if denom > 0 else 0.0
    scores[e] = {"precision": precision, "recall": recall, f"f{BETA}": f_score,
                 "tp": _tp, "fp": _fp, "fn": _fn}

import pandas as pd
scores_df = pd.DataFrame(scores).T.round(3)
print(scores_df)

/Users/nikoli/Desktop/IC_LLM_API_GW/pii_detection_test/venv/lib/python3.11/site-packages/torch/nn/modules/module.py:1527: FutureWarning: `encoder_attention_mask` is deprecated and will be removed in version 4.55.0 for `ElectraSelfAttention.forward`.
  return forward_call(*args, **kwargs)


               precision  recall     f2      tp       fp     fn
EMAIL_ADDRESS      0.000   0.000  0.000     0.0      0.0  974.0
ID                 0.000   0.000  0.000     0.0      0.0  975.0
LOCATION           0.000   0.000  0.000     0.0      0.0  975.0
PERSON             0.441   0.962  0.350  2791.0   3539.0  110.0
PHONE_NUMBER       0.051   0.666  0.058   649.0  12113.0  325.0


In [6]:
import plotly.graph_objects as go
from pathlib import Path

MODEL_NAME = "HanLP ELECTRA small ZH"
output_folder = Path(PROJECT_DIR, "plotter_output")
output_folder.mkdir(exist_ok=True)

# ── Per-entity bar chart (same layout as Presidio Plotter) ──────────────────
for etype, s in scores.items():
    fig = go.Figure()
    fig.add_bar(name="Precision", x=["Precision"], y=[s["precision"]],
                marker_color="#1f77b4", text=[f"{s['precision']:.2f}"], textposition="outside")
    fig.add_bar(name="Recall",    x=["Recall"],    y=[s["recall"]],
                marker_color="#ff7f0e", text=[f"{s['recall']:.2f}"],    textposition="outside")
    fig.add_bar(name=f"F{BETA}",  x=[f"F{BETA}"],  y=[s[f'f{BETA}']],
                marker_color="#2ca02c", text=[f"{s[f'f{BETA}']:.2f}"], textposition="outside")
    fig.update_layout(
        title=f"{MODEL_NAME} — {etype}  (TP={s['tp']}, FP={s['fp']}, FN={s['fn']})",
        yaxis=dict(range=[0, 1.1], title="Score"),
        barmode="group", width=500, height=400,
        legend=dict(orientation="h", y=-0.15),
    )
    fig.show()

# ── Summary chart — all entities side-by-side ───────────────────────────────
entities  = list(scores.keys())
precision = [scores[e]["precision"] for e in entities]
recall    = [scores[e]["recall"]    for e in entities]
f_scores  = [scores[e][f"f{BETA}"] for e in entities]

fig_all = go.Figure()
fig_all.add_bar(name="Precision", x=entities, y=precision,
                marker_color="#1f77b4",
                text=[f"{v:.2f}" for v in precision], textposition="outside")
fig_all.add_bar(name="Recall",    x=entities, y=recall,
                marker_color="#ff7f0e",
                text=[f"{v:.2f}" for v in recall],    textposition="outside")
fig_all.add_bar(name=f"F{BETA}", x=entities, y=f_scores,
                marker_color="#2ca02c",
                text=[f"{v:.2f}" for v in f_scores],  textposition="outside")
fig_all.update_layout(
    title=f"{MODEL_NAME} — All Entities (β={BETA})",
    yaxis=dict(range=[0, 1.2], title="Score"),
    barmode="group", width=900, height=500,
    legend=dict(orientation="h", y=-0.15),
)
fig_all.show()

In [7]:
# Overall PII-level F / recall / precision
all_tp = sum(tp.values())
all_fp = sum(fp.values())
all_fn = sum(fn.values())
pii_precision = all_tp / (all_tp + all_fp) if (all_tp + all_fp) > 0 else 0
pii_recall    = all_tp / (all_tp + all_fn) if (all_tp + all_fn) > 0 else 0
denom = (1 + BETA**2) * pii_precision + BETA**2 * pii_recall
pii_f = (1 + BETA**2) * pii_precision * pii_recall / denom if denom > 0 else 0
from pprint import pprint
pprint({f"PII F{BETA}": round(pii_f, 3),
        "PII recall": round(pii_recall, 3),
        "PII precision": round(pii_precision, 3)})

{'PII F2': 0.147, 'PII precision': 0.166, 'PII recall': 0.506}


## 9. Multi-dataset evaluation with language-specific analyzers

This section evaluates all three datasets with the correct analyzer by language:
- `data_person_1000` -> English analyzer (`language="en"`)
- `data_person_1000_zh` -> HanLP Chinese analyzer (`language="zh"`)
- `mixed_pii_code_switch_500` -> HanLP Chinese analyzer (`language="zh"`)

Outputs:
- Per-entity Precision / Recall / F2 for each dataset
- Overall PII Precision / Recall / F2 across datasets

Mixed-label normalization:
- `PERSON_CN` / `PERSON_EN` -> `PERSON`
- `EMAIL` -> `EMAIL_ADDRESS`
- `PHONE` -> `PHONE_NUMBER`
- `CN_ID` / `PASSPORT` -> `ID`
- `ADDRESS_CN` / `ADDRESS_EN` -> `LOCATION`

In [10]:
from collections import defaultdict
import pandas as pd
import plotly.graph_objects as go

from presidio_analyzer.nlp_engine import NlpEngineProvider

DATASET_CONFIGS = [
    {
        "name": "data_person_1000",
        "path": PROJECT_DIR / "pii_test_data" / "data_person_1000.json",
        "schema": "person1000",
        "lang": "en",
    },
    {
        "name": "data_person_1000_zh",
        "path": PROJECT_DIR / "pii_test_data" / "data_person_1000_zh.json",
        "schema": "person1000",
        "lang": "zh",
    },
    {
        "name": "mixed_pii_code_switch_500",
        "path": PROJECT_DIR / "pii_test_data" / "mixed_pii_code_switch_500.json",
        "schema": "mixed500",
        "lang": "zh",
    },
]

PERSON1000_FIELD_MAP = {
    "name": "PERSON",
    "doctor": "PERSON",
    "emailAddress": "EMAIL_ADDRESS",
    "phoneNumbers": "PHONE_NUMBER",
    "location": "LOCATION",
    "idCardNumbers": "ID",
}

MIXED500_LABEL_MAP = {
    "PERSON_CN": "PERSON",
    "PERSON_EN": "PERSON",
    "EMAIL": "EMAIL_ADDRESS",
    "PHONE": "PHONE_NUMBER",
    "CN_ID": "ID",
    "PASSPORT": "ID",
    "ADDRESS_CN": "LOCATION",
    "ADDRESS_EN": "LOCATION",
}


def overlap(a_start, a_end, b_start, b_end):
    return a_start < b_end and b_start < a_end


def build_en_analyzer():
    config = {
        "nlp_engine_name": "spacy",
        "models": [{"lang_code": "en", "model_name": "en_core_web_sm"}],
    }
    provider = NlpEngineProvider(nlp_configuration=config)
    en_nlp_engine = provider.create_engine()
    return AnalyzerEngine(nlp_engine=en_nlp_engine, supported_languages=["en"])


try:
    analyzer_en = build_en_analyzer()
except Exception as e:
    raise RuntimeError(
        "English model is not ready. Install it in your venv with:\n"
        "python -m spacy download en_core_web_sm\n"
        f"Original error: {e}"
    )


def analyze_with_lang(text, lang):
    if lang == "en":
        return analyzer_en.analyze(text=text, language="en", entities=SUPPORTED_ENTITIES)
    return analyzer.analyze(text=text, language="zh", entities=SUPPORTED_ENTITIES)


results_by_dataset = {}
rows_entity_scores = []
rows_pii_scores = []

for cfg in DATASET_CONFIGS:
    with open(cfg["path"], "r", encoding="utf-8") as f:
        ds_data = json.load(f)

    tp = defaultdict(int)
    fp = defaultdict(int)
    fn = defaultdict(int)

    for row in ds_data:
        if cfg["schema"] == "person1000":
            text = row.get("naturalParagraph", "")
            if not text:
                continue

            gt_spans = []
            for field, etype in PERSON1000_FIELD_MAP.items():
                raw = row.get(field)
                if raw is None:
                    continue
                value = str(raw).strip()
                if not value:
                    continue

                start = 0
                while True:
                    idx = text.find(value, start)
                    if idx == -1:
                        break
                    gt_spans.append((idx, idx + len(value), etype))
                    start = idx + 1

        elif cfg["schema"] == "mixed500":
            text = row.get("text", "")
            if not text:
                continue

            gt_spans = []
            for item in row.get("pii_labels", []):
                label = item.get("label")
                etype = MIXED500_LABEL_MAP.get(label)
                if etype is None:
                    continue

                start = item.get("start")
                end = item.get("end")
                if isinstance(start, int) and isinstance(end, int) and start < end:
                    gt_spans.append((start, end, etype))

        else:
            continue

        pred_results = analyze_with_lang(text=text, lang=cfg["lang"])
        pred_spans = [(r.start, r.end, r.entity_type) for r in pred_results]

        matched_gt = set()
        for ps, pe, ptype in pred_spans:
            matched = False
            for gi, (gs, ge, gtype) in enumerate(gt_spans):
                if gi in matched_gt:
                    continue
                if ptype == gtype and overlap(ps, pe, gs, ge):
                    tp[ptype] += 1
                    matched_gt.add(gi)
                    matched = True
                    break
            if not matched:
                fp[ptype] += 1

        for gi, (_, _, gtype) in enumerate(gt_spans):
            if gi not in matched_gt:
                fn[gtype] += 1

    entity_types = sorted(set(list(tp.keys()) + list(fn.keys()) + list(fp.keys())))
    ds_scores = {}

    for etype in entity_types:
        _tp, _fp, _fn = tp[etype], fp[etype], fn[etype]
        precision = _tp / (_tp + _fp) if (_tp + _fp) > 0 else 0.0
        recall = _tp / (_tp + _fn) if (_tp + _fn) > 0 else 0.0
        denom = (1 + BETA**2) * precision + BETA**2 * recall
        f_score = (1 + BETA**2) * precision * recall / denom if denom > 0 else 0.0

        ds_scores[etype] = {
            "precision": precision,
            "recall": recall,
            f"f{BETA}": f_score,
            "tp": _tp,
            "fp": _fp,
            "fn": _fn,
        }

        rows_entity_scores.append(
            {
                "dataset": cfg["name"],
                "language": cfg["lang"],
                "entity": etype,
                "precision": round(precision, 3),
                "recall": round(recall, 3),
                f"f{BETA}": round(f_score, 3),
                "tp": _tp,
                "fp": _fp,
                "fn": _fn,
            }
        )

    all_tp = sum(tp.values())
    all_fp = sum(fp.values())
    all_fn = sum(fn.values())
    pii_precision = all_tp / (all_tp + all_fp) if (all_tp + all_fp) > 0 else 0.0
    pii_recall = all_tp / (all_tp + all_fn) if (all_tp + all_fn) > 0 else 0.0
    pii_denom = (1 + BETA**2) * pii_precision + BETA**2 * pii_recall
    pii_f = (1 + BETA**2) * pii_precision * pii_recall / pii_denom if pii_denom > 0 else 0.0

    rows_pii_scores.append(
        {
            "dataset": cfg["name"],
            "language": cfg["lang"],
            "records": len(ds_data),
            "pii_precision": round(pii_precision, 3),
            "pii_recall": round(pii_recall, 3),
            f"pii_f{BETA}": round(pii_f, 3),
            "tp": all_tp,
            "fp": all_fp,
            "fn": all_fn,
        }
    )

    results_by_dataset[cfg["name"]] = {
        "language": cfg["lang"],
        "records": len(ds_data),
        "entity_scores": ds_scores,
        "pii": {"precision": pii_precision, "recall": pii_recall, f"f{BETA}": pii_f},
    }

entity_scores_df = pd.DataFrame(rows_entity_scores).sort_values(["dataset", "entity"]).reset_index(drop=True)
pii_scores_df = pd.DataFrame(rows_pii_scores).sort_values(["dataset"]).reset_index(drop=True)

print("Per-entity scores:")
display(entity_scores_df)
print("Overall PII scores:")
display(pii_scores_df)

# Per-dataset summary chart (all entities)
for ds_name in pii_scores_df["dataset"].tolist():
    ds_rows = entity_scores_df[entity_scores_df["dataset"] == ds_name]
    if ds_rows.empty:
        continue

    ds_lang = ds_rows["language"].iloc[0]

    fig = go.Figure()
    fig.add_bar(name="Precision", x=ds_rows["entity"], y=ds_rows["precision"], marker_color="#1f77b4")
    fig.add_bar(name="Recall", x=ds_rows["entity"], y=ds_rows["recall"], marker_color="#ff7f0e")
    fig.add_bar(name=f"F{BETA}", x=ds_rows["entity"], y=ds_rows[f"f{BETA}"], marker_color="#2ca02c")
    fig.update_layout(
        title=f"HanLP/Presidio - {ds_name} (lang={ds_lang}, entity-level)",
        yaxis=dict(range=[0, 1.1], title="Score"),
        barmode="group",
        width=950,
        height=500,
        legend=dict(orientation="h", y=-0.18),
    )
    fig.show()

# Cross-dataset overall PII comparison
fig_cmp = go.Figure()
fig_cmp.add_bar(name="PII Precision", x=pii_scores_df["dataset"], y=pii_scores_df["pii_precision"], marker_color="#1f77b4")
fig_cmp.add_bar(name="PII Recall", x=pii_scores_df["dataset"], y=pii_scores_df["pii_recall"], marker_color="#ff7f0e")
fig_cmp.add_bar(name=f"PII F{BETA}", x=pii_scores_df["dataset"], y=pii_scores_df[f"pii_f{BETA}"], marker_color="#2ca02c")
fig_cmp.update_layout(
    title=f"HanLP/Presidio - PII comparison across datasets (beta={BETA})",
    yaxis=dict(range=[0, 1.1], title="Score"),
    barmode="group",
    width=900,
    height=500,
    legend=dict(orientation="h", y=-0.18),
)
fig_cmp.show()

compare_output_path = PROJECT_DIR / "pii_test_data" / "hanlp_presidio_dataset_comparison.json"
with open(compare_output_path, "w", encoding="utf-8") as f:
    json.dump(
        {
            "beta": BETA,
            "datasets": results_by_dataset,
            "entity_scores": rows_entity_scores,
            "pii_scores": rows_pii_scores,
        },
        f,
        ensure_ascii=False,
        indent=2,
    )

print("Saved comparison JSON:", compare_output_path)

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 5.5 MB/s  0:00:02 eta 0:00:01
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


Exception reading Public Suffix List url https://publicsuffix.org/list/public_suffix_list.dat
Traceback (most recent call last):
  File "/Users/nikoli/Desktop/IC_LLM_API_GW/pii_detection_test/venv/lib/python3.11/site-packages/tldextract/cache.py", line 196, in run_and_cache
    result = cast(T, self.get(namespace=namespace, key=key_args))
                     ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/Users/nikoli/Desktop/IC_LLM_API_GW/pii_detection_test/venv/lib/python3.11/site-packages/tldextract/cache.py", line 99, in get
    raise KeyError("namespace: " + namespace + " key: " + repr(key))
KeyError: "namespace: publicsuffix.org-tlds key: {'urls': ('https://publicsuffix.org/list/public_suffix_list.dat', 'https://raw.githubusercontent.com/publicsuffix/list/master/public_suffix_list.dat'), 'fallback_to_snapshot': True}"

During handling of the above exception, another exception occurred:

Traceback (most recent call last):
  File "/Users/nikoli/Desktop/IC_LLM_API_GW/pii_detec

Per-entity scores:


,dataset,language,entity,precision,recall,f2,tp,fp,fn
0,data_person_1000,en,DATE_TIME,0.000,0.000,0.000,0,3588,0
1,data_person_1000,en,EMAIL_ADDRESS,0.975,1.000,0.549,974,25,0
2,data_person_1000,en,ID,0.000,0.000,0.000,0,0,999
3,data_person_1000,en,LOCATION,0.057,0.068,0.035,67,1100,922
4,data_person_1000,en,ORGANIZATION,0.000,0.000,0.000,0,908,0
5,data_person_1000,en,PERSON,0.317,0.970,0.281,1016,2193,31
6,data_person_1000,en,PHONE_NUMBER,0.990,0.766,0.473,765,8,234
7,data_person_1000,en,URL,0.000,0.000,0.000,0,1205,0
8,data_person_1000_zh,zh,DATE_TIME,0.000,0.000,0.000,0,11,0
9,data_person_1000_zh,zh,EMAIL_ADDRESS,0.000,0.000,0.000,0,0,974


Overall PII scores:


,dataset,language,records,pii_precision,pii_recall,pii_f2,tp,fp,fn
0,data_person_1000,en,1000,0.238,0.563,0.195,2822,9027,2186
1,data_person_1000_zh,zh,975,0.166,0.506,0.147,3440,17224,3359
2,mixed_pii_code_switch_500,zh,500,0.092,0.250,0.078,768,7621,2310


Saved comparison JSON: /Users/nikoli/Desktop/IC_LLM_API_GW/pii_detection_test/pii_test_data/hanlp_presidio_dataset_comparison.json


In [11]:
# HanLP-only benchmark on English dataset (force zh analyzer on data_person_1000)
from collections import defaultdict

en_path = PROJECT_DIR / "pii_test_data" / "data_person_1000.json"
with open(en_path, "r", encoding="utf-8") as f:
    en_data = json.load(f)

field_map_en = {
    "name": "PERSON",
    "doctor": "PERSON",
    "emailAddress": "EMAIL_ADDRESS",
    "phoneNumbers": "PHONE_NUMBER",
    "location": "LOCATION",
    "idCardNumbers": "ID",
}

def _find_all_spans(text, value):
    spans, start = [], 0
    while True:
        idx = text.find(value, start)
        if idx == -1:
            break
        spans.append((idx, idx + len(value)))
        start = idx + 1
    return spans

def _overlap(a_start, a_end, b_start, b_end):
    return a_start < b_end and b_start < a_end

tp_h, fp_h, fn_h = defaultdict(int), defaultdict(int), defaultdict(int)
for row in en_data:
    text = row.get("naturalParagraph", "")
    if not text:
        continue

    gt_spans = []
    for field, etype in field_map_en.items():
        raw = row.get(field)
        if raw is None:
            continue
        value = str(raw).strip()
        if not value:
            continue
        for s, e in _find_all_spans(text, value):
            gt_spans.append((s, e, etype))

    # Force HanLP path for this benchmark
    pred_results = analyzer.analyze(text=text, language="zh", entities=SUPPORTED_ENTITIES)
    pred_spans = [(r.start, r.end, r.entity_type) for r in pred_results]

    matched_gt = set()
    for ps, pe, ptype in pred_spans:
        matched = False
        for gi, (gs, ge, gtype) in enumerate(gt_spans):
            if gi in matched_gt:
                continue
            if ptype == gtype and _overlap(ps, pe, gs, ge):
                tp_h[ptype] += 1
                matched_gt.add(gi)
                matched = True
                break
        if not matched:
            fp_h[ptype] += 1

    for gi, (_, _, gtype) in enumerate(gt_spans):
        if gi not in matched_gt:
            fn_h[gtype] += 1

all_tp_h = sum(tp_h.values())
all_fp_h = sum(fp_h.values())
all_fn_h = sum(fn_h.values())

pii_precision_h = all_tp_h / (all_tp_h + all_fp_h) if (all_tp_h + all_fp_h) > 0 else 0.0
pii_recall_h = all_tp_h / (all_tp_h + all_fn_h) if (all_tp_h + all_fn_h) > 0 else 0.0
denom_h = (1 + BETA**2) * pii_precision_h + BETA**2 * pii_recall_h
pii_f2_h = (1 + BETA**2) * pii_precision_h * pii_recall_h / denom_h if denom_h > 0 else 0.0

print("HanLP-only on data_person_1000 (forced zh analyzer):")
print({
    "records": len(en_data),
    "pii_precision": round(pii_precision_h, 3),
    "pii_recall": round(pii_recall_h, 3),
    f"pii_f{BETA}": round(pii_f2_h, 3),
    "tp": all_tp_h,
    "fp": all_fp_h,
    "fn": all_fn_h,
})

/Users/nikoli/Desktop/IC_LLM_API_GW/pii_detection_test/venv/lib/python3.11/site-packages/torch/nn/modules/module.py:1527: FutureWarning:

`encoder_attention_mask` is deprecated and will be removed in version 4.55.0 for `ElectraSelfAttention.forward`.



HanLP-only on data_person_1000 (forced zh analyzer):
{'records': 1000, 'pii_precision': 0.171, 'pii_recall': 0.102, 'pii_f2': 0.069, 'tp': 509, 'fp': 2466, 'fn': 4499}
